# Feature Engineering:
| Original Variable       | New Feature                    | Categories                                    | Purpose                                    |
| ----------------------- | ------------------------------ | --------------------------------------------- | ------------------------------------------ |
| **Age**                 | `Age_Group`                    | Young Adult, Middle-aged, Older Adult, Senior | Examine readmission differences by age     |
| **Length of Stay**      | `LOS_Category`                 | Short, Moderate, Long, Extended               | Identify hospitalization-duration patterns |
| **Previous Admissions** | `Previous_Admissions_Category` | None, Low, Moderate, High                     | Represent previous hospital utilization    |
| **Medication Count**    | `Medication_Burden`            | Low, Moderate, High                           | Represent medication burden                |
| **Treatment Cost**      | `Cost_Category`                | Low, Moderate, High, Very High                | Compare readmission patterns by cost       |
| **Previous ER Visits**  | `ER_Visit_Category`            | None, Low, Moderate, High                     | Represent previous emergency utilization   |


# Loading the dataset:


In [1]:
import pandas as pd

cleaned_data = pd.read_csv(
    r"C:\Users\Hi\Desktop\Risk-Analysis\data\processed\ProcessedData.csv"
)

print("Cleaned dataset loaded successfully!")
print(cleaned_data.shape)

Cleaned dataset loaded successfully!
(500, 23)


# Making a separate copy for feature engineering


In [2]:
feature_data = cleaned_data.copy()

Checking original columns  

In [3]:
print(feature_data.columns.tolist())

['Patient_ID', 'Age', 'Gender', 'Region', 'Insurance_Type', 'Admission_Type', 'Hospital_Department', 'Length_of_Stay', 'Previous_Admissions', 'Previous_ER_Visits', 'Diabetes', 'Hypertension', 'Heart_Disease', 'Medication_Count', 'Lab_Test_Count', 'Average_Glucose', 'Systolic_BP', 'Discharge_Type', 'Followup_Scheduled', 'Followup_Attended', 'Treatment_Cost', 'Satisfaction_Score', 'Readmitted_30_Days']


# Adding Engineered Features

1. Age Group:


In [4]:
feature_data["Age_Group"] = pd.cut(
    feature_data["Age"],
    bins=[17, 34, 49, 64, float("inf")],
    labels=["Young Adult", "Middle-aged", "Older Adult", "Senior"]
)

2.Length of Stay Category:


In [5]:
feature_data["LOS_Category"] = pd.cut(
    feature_data["Length_of_Stay"],
    bins=[0, 3, 7, 14, float("inf")],
    labels=["Short Stay", "Moderate Stay", "Long Stay", "Extended Stay"]
)

3. Previous Admission Category:

In [6]:
feature_data["Previous_Admissions_Category"] = pd.cut(
    feature_data["Previous_Admissions"],
    bins=[-1, 0, 2, 5, float("inf")],
    labels=["None", "Low", "Moderate", "High"]
)

4. Medication Burden:

In [7]:
feature_data["Medication_Burden"] = pd.cut(
    feature_data["Medication_Count"],
    bins=[-1, 2, 5, float("inf")],
    labels=["Low", "Moderate", "High"]
)


5. Previous ER Visits

In [8]:
feature_data["ER_Visit_Category"] = pd.cut(
    feature_data["Previous_ER_Visits"],
    bins=[-1, 0, 2, 5, float("inf")],
    labels=["None", "Low", "Moderate", "High"]
)

# Checking Variables

In [9]:
print("Cleaned data:", cleaned_data.shape)
print("Feature data:", feature_data.shape)

Cleaned data: (500, 23)
Feature data: (500, 28)


# Saving the feature data

In [10]:
output_path = r"C:\Users\Hi\Desktop\Risk-Analysis\data\processed\EngineeredData.csv"

feature_data.to_csv(output_path, index=False)

print("Feature-engineered dataset saved successfully!")

Feature-engineered dataset saved successfully!


# Profiling Each new feature against Readmitted 30 days


# Reusable Profiling Function


In [15]:
# creating a temporary numeric variable for target variable
feature_data["Readmitted_Flag"] = (
    feature_data["Readmitted_30_Days"]
    .map({"Yes": 1, "No": 0})
)

In [16]:
def profile_categorical_feature(feature_data, feature, target="Readmitted_Flag"):
    
    profile = (
        feature_data.groupby(feature, observed=True)[target]
        .agg(
            Total_Patients="count",
            Readmitted="sum",
            Readmission_Rate="mean"
        )
        .reset_index()
    )
    
    profile["Readmission_Rate"] = (
        profile["Readmission_Rate"] * 100
    ).round(2)
    
    return profile

# Testing for Each variable:
Age Group


In [17]:
age_profile = profile_categorical_feature(
    feature_data,
    "Age_Group"
)

print(age_profile)


     Age_Group  Total_Patients  Readmitted  Readmission_Rate
0  Young Adult             113          22             19.47
1  Middle-aged              99          17             17.17
2  Older Adult             110          28             25.45
3       Senior             178          53             29.78


2. Length of Stay

In [18]:
los_profile = profile_categorical_feature(
    feature_data,
    "LOS_Category"
)

print(los_profile)

    LOS_Category  Total_Patients  Readmitted  Readmission_Rate
0     Short Stay             235          52             22.13
1  Moderate Stay             191          46             24.08
2      Long Stay              69          19             27.54
3  Extended Stay               5           3             60.00


3. Previous Admissions

In [19]:
admission_profile = profile_categorical_feature(
    feature_data,
    "Previous_Admissions_Category"
)

print(admission_profile)

  Previous_Admissions_Category  Total_Patients  Readmitted  Readmission_Rate
0                         None             155          26             16.77
1                          Low             294          73             24.83
2                     Moderate              51          21             41.18


4. Medication Burden


In [20]:
medication_profile = profile_categorical_feature(
    feature_data,
    "Medication_Burden"
)

print(medication_profile)

  Medication_Burden  Total_Patients  Readmitted  Readmission_Rate
0               Low              37          10             27.03
1          Moderate             188          46             24.47
2              High             275          64             23.27


5. Previous ER Visits

In [21]:
er_profile = profile_categorical_feature(
    feature_data,
    "ER_Visit_Category"
)

print(er_profile)

  ER_Visit_Category  Total_Patients  Readmitted  Readmission_Rate
0              None             114          27             23.68
1               Low             302          66             21.85
2          Moderate              82          25             30.49
3              High               2           2            100.00


# Summary Table Report:

# Statistical Association



In [30]:
# Reusable test function
from sklearn.feature_selection import chi2
from scipy.stats import chi2_contingency


def chi_square_test(feature_data, feature, target="Readmitted_30_Days"):
    
    table = pd.crosstab(
        feature_data[feature],
        feature_data[target]
    )
    
    chi2, p_value, dof, expected = chi2_contingency(table)
    
    return {
        "Feature": feature,
        "Chi-square": round(chi2, 3),
        "p-value": round(p_value, 4)
    }

In [31]:

features = [
    "Age_Group",
    "LOS_Category",
    "Previous_Admissions_Category",
    "Medication_Burden",
    "ER_Visit_Category"
]

results = []

for feature in features:
    results.append(
        chi_square_test(feature_data, feature)
    )

chi_square_results = pd.DataFrame(results)

print(chi_square_results)

                        Feature  Chi-square  p-value
0                     Age_Group       7.185   0.0662
1                  LOS_Category       4.478   0.2143
2  Previous_Admissions_Category      12.797   0.0017
3             Medication_Burden       0.288   0.8658
4             ER_Visit_Category       8.994   0.0294


# Conclusion:
there is statistical significance evidence of an association between the Previous_admissions_category, ER_Visits_category and 30-day readmission.